<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.2: Similitud de Coseno e Indice Invertido

> **Prof. Heider Sanchez**  

## Introducción

Este laboratorio extiende las capacidades desarrolladas en el laboratorio 7.1, enfocándose en técnicas avanzadas de recuperación de información. Utilizaremos los Bag of Words previamente generados para implementar dos funcionalidades esenciales en los motores de búsqueda modernos: el **Índice Invertido** para recuperación eficiente de documentos, y la **Similitud de Coseno** para resultados ordenados por relevancia.


### Objetivos
- Implementar la conexión a PostgreSQL para extraer id, contenido y bag_of_words de los documentos.
- Construir el índice invertido (posting lists), calcular estadísticas IDF y la norma vectorial de cada documento.
- Implementar búsquedas booleanas (AND, OR y AND-NOT) con complejidad O(n+m) usando el índice invertido.
- Implementar búsquedas rankeadas usando la Similitud de Coseno:
  - Procesar consultas en lenguaje natural aplicando tokenización y cálculo del TF.
  - Utilizar el índice invertido para obtener los documentos que intersectan con la query.
  - Calcular similitud de coseno con TF-IDF entre la query y los documentos recuperados.
  - Devolver los top-k resultados ordenados por relevancia.
  - **Evitar usar la representación vectorial dispersa.**

In [1]:
import psycopg2
import pandas as pd
import json 

def connect_db():
    conn = psycopg2.connect(
        dbname="lab8",
        user="postgres",
        password="postgres",
        host="localhost"
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido, bag_of_words FROM noticias;"
    df = pd.read_sql(query, conn)
    #df['bag_of_words'] = df['bag_of_words'].apply(json.loads)
    conn.close()
    return df

noticias_df = fetch_data()
noticias_df.head()

C:\Users\Paris Herrera\AppData\Local\Temp\ipykernel_23952\2296358644.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,contenido,bag_of_words
0,8,Durante el foro La banca articulador empresari...,"{'aca': 1, 'adn': 1, 'cas': 1, 'for': 1, 'ret'..."
1,9,El regulador de valores de China dijo el domin...,"{'dat': 1, 'deb': 1, 'inc': 1, 'med': 1, 'mes'..."
2,10,En una industria históricamente masculina como...,"{'go': 1, 'air': 1, 'baj': 1, 'ceo': 2, 'hij':..."
3,11,Con el dato de marzo el IPC interanual encaden...,"{'agu': 1, 'car': 2, 'cas': 1, 'dat': 2, 'ine'..."
4,12,Ayer en Cartagena se dio inicio a la versión n...,"{'agu': 1, 'cup': 2, 'deb': 2, 'etc': 1, 'for'..."


## 1. (6 puntos) Construcción del Indice Invertido 

A partir de los  `bag of words` almacenados en la base de datos  (Laboratorio 8.1), se debe construir un índice invertido y conservarlo en un diccionario de Python para su eficiente recuperación.

In [2]:
import math
import re
from collections import Counter
from nltk.stem import SnowballStemmer


#Funciones auziliares para procesamiento de la consulta 
def fetch_stopwords():
    conn = connect_db()
    
    query = """
        SELECT word
        FROM stopwords;
    """
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    stop_words = set(
        df["word"]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
    )
    
    return stop_words

stop_words = fetch_stopwords()
stemmer = SnowballStemmer("spanish")

def preprocess(text):
    text = str(text)
    text = text.lower()
    
    tokens = re.findall(r"\b[a-záéíóúñü]+\b", text)
    
    tokens = [
        token for token in tokens
        if token not in stop_words
    ]
    
    tokens = [
        stemmer.stem(token)
        for token in tokens
    ]
    
    return tokens

def compute_bow(text):
    tokens = preprocess(text)
    bow = Counter(tokens)
    return dict(bow)

class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.idf = {}
        self.length = {}

    def build_from_db(self):

        # Limpiamos las estructuras
        self.index = {}
        self.idf = {}
        self.length = {}

        N = len(noticias_df)

        #Construir índice invertido
        for _, row in noticias_df.iterrows():
            doc_id = row["id"]
            bag_of_words = row["bag_of_words"]

            for word, tf in bag_of_words.items():
                if word not in self.index:
                    self.index[word] = []

                self.index[word].append((doc_id, tf))

        #Ordenar para consultas booleanas 
        for word in self.index:
            self.index[word].sort(key=lambda tup: tup[0])


        #Calcular IDF
        for word, posting_list in self.index.items():
            df_word = len(posting_list)
            self.idf[word] = math.log10(N / df_word)


        #Calcular longitud/norma de cada documento
        for _, row in noticias_df.iterrows():
            doc_id = row["id"]
            bag_of_words = row["bag_of_words"]

            suma_cuadrados = 0

            for word, tf in bag_of_words.items():
                tf_weight = 1 + math.log10(tf)
                tf_idf = tf_weight * self.idf[word]

                suma_cuadrados += tf_idf ** 2

            self.length[doc_id] = math.sqrt(suma_cuadrados)
        
    
        """
        indice  = {
            "word1": [("doc1", tf1), ("doc2", tf2), ("doc3", tf3)],
            "word2": [("doc2", tf2), ("doc4", tf4)],
            "word3": [("doc3", tf3), ("doc5", tf5)],
        } 
        idf  = {
            "word1": 3,
            "word2": 2,
            "word3": 2,
        } 
        length = {
            "doc1": 15.5236,
            "doc2": 10.5236,
            "doc3": 5.5236,
        }
        """
        pass
    
    def L(self, word):
        # Retorna la lista de documentos que contienen a word 
        return self.index.get(word, [])
  
    def cosine_search(self, query, top_k=5):  

        #Procesar Query
        query_bow = compute_bow(query)


        #Calcular pesos TF-IDF de la query
        query_weights = {}

        for word, tf in query_bow.items():
            if word in self.idf:
                tf_query = 1 + math.log10(tf)
                query_weights[word] = tf_query * self.idf[word]

        #Calcular norma de la query
        query_length = math.sqrt(sum(weight ** 2 for weight in query_weights.values()))

        if query_length == 0:
            return []
        

        #Acumular producto punto usando el índice invertido
        score = {}

        for word, query_weight in query_weights.items():
            posting_list = self.L(word)

            for doc_id, tf_doc in posting_list:
                tf_doc_weight = 1 + math.log10(tf_doc)
                doc_weight = tf_doc_weight * self.idf[word]

                if doc_id not in score:
                    score[doc_id] = 0

                score[doc_id] += query_weight * doc_weight


        #Normalizar para obtener similitud de coseno
        for doc_id in score:
            doc_length = self.length[doc_id]

            if doc_length != 0:
                score[doc_id] = score[doc_id] / (query_length * doc_length)

        
        # Ordenar el score resultante de forma descendente
        result = sorted(score.items(), key= lambda tup: tup[1], reverse=True)
        # retornamos los k documentos mas relevantes (de mayor similitud a la query)
        return result[:top_k] 
    
    def showDocuments(self, result):
        doc_ids = [doc_id for doc_id, _ in result]
        return doc_ids

    def showDocument2(self, doc_id):
        fila = noticias_df[noticias_df["id"] == doc_id]

        if fila.empty:
            return "Documento no encontrado"

        contenido = fila.iloc[0]["contenido"]

        return contenido[:100] + "..."

C:\Users\Paris Herrera\AppData\Local\Temp\ipykernel_23952\2704021397.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 2. (6 puntos) Consultas Booleanas usando el indice invertido

Implementar búsquedas booleanas utilizando el índice invertido construido anteriormente. La búsqueda debe:

- Soportar los operadores básicos:
    - AND: intersección de documentos
    - OR: unión de documentos
    - AND-NOT: diferencia de documentos
- Procesar consultas como:
    - "sostenibilidad AND ambiente AND renovable"
    - "tecnología AND (banca OR finanzas)"
    - "economía AND-NOT inflación"    

####  Pruebas funcionales

In [3]:
idx = InvertedIndex()
idx.build_from_db()

def AND(list1, list2):
    result = []

    i = 0
    j = 0

    while i < len(list1) and j < len(list2):
        doc1 = list1[i][0]
        doc2 = list2[j][0]

        if doc1 == doc2:
            result.append(list1[i])
            i += 1
            j += 1
        elif doc1 < doc2:
            i += 1
        else:
            j += 1

    return result

def OR(list1, list2):
    result = []

    i = 0
    j = 0

    while i < len(list1) and j < len(list2):
        doc1 = list1[i][0]
        doc2 = list2[j][0]

        if doc1 == doc2:
            result.append(list1[i])
            i += 1
            j += 1
        elif doc1 < doc2:
            result.append(list1[i])
            i += 1
        else:
            result.append(list2[j])
            j += 1

    while i < len(list1):
        result.append(list1[i])
        i += 1

    while j < len(list2):
        result.append(list2[j])
        j += 1

    return result

def AND_NOT(list1, list2):
    result = []

    i = 0
    j = 0

    while i < len(list1) and j < len(list2):
        doc1 = list1[i][0]
        doc2 = list2[j][0]

        if doc1 == doc2:
            i += 1
            j += 1
        elif doc1 < doc2:
            result.append(list1[i])
            i += 1
        else:
            j += 1

    while i < len(list1):
        result.append(list1[i])
        i += 1

    return result

#Funcion para procesar las palabras solas, como la query 
def term(word):
    tokens = preprocess(word)

    if len(tokens) == 0:
        return ""

    return tokens[0]


# Prueba 1
result1 = AND(idx.L(term("sostenibilidad")),     AND(idx.L(term("ambiente")),     idx.L(term("renovables"))))
print("sostenibilidad AND ambiente AND renovable: \n", idx.showDocuments(result1),  "\n")

# Prueba 2
result2 = AND(idx.L(term("tecnología")),      OR(idx.L(term("banca")),      idx.L(term("finanzas"))))
print("tecnología AND (banca OR finanzas): \n", idx.showDocuments(result2), "\n")

# Prueba 3
result3 = AND_NOT(     idx.L(term("economía")),      idx.L(term("inflación")))
print("economía AND-NOT inflación: \n" , idx.showDocuments(result3), "\n")



# Agregar dos pruebas mas combinando los operadores AND, OR, AND_NOT
result4 = OR( AND( idx.L(term("banco")), idx.L(term("demostraciones")) ),  idx.L(term("app")) )
print("OR( AND(banca, demostraciones), app): \n" ,  idx.showDocuments(result4), "\n")

result5 = AND_NOT( OR( idx.L(term("Italia")), idx.L(term("economia")) ),  idx.L(term("BBVA")) )
print("AND_NOT( OR(BBVA, CO2), economia): \n" ,  idx.showDocuments(result5), "\n")



sostenibilidad AND ambiente AND renovable: 
 [59, 404, 427, 476, 560, 644, 1133, 1172, 1186] 

tecnología AND (banca OR finanzas): 
 [19, 20, 22, 23, 24, 27, 37, 56, 59, 62, 70, 85, 92, 95, 100, 118, 123, 142, 151, 154, 163, 185, 186, 194, 200, 201, 212, 213, 217, 219, 233, 235, 241, 244, 253, 260, 263, 265, 266, 276, 289, 293, 300, 309, 319, 328, 339, 365, 369, 372, 376, 383, 398, 401, 413, 422, 423, 424, 426, 437, 452, 462, 463, 467, 472, 485, 505, 514, 518, 529, 547, 550, 578, 580, 590, 591, 601, 605, 608, 613, 614, 618, 624, 630, 633, 635, 640, 646, 649, 651, 662, 672, 676, 681, 682, 686, 693, 696, 703, 712, 726, 730, 745, 755, 788, 793, 800, 826, 833, 836, 847, 849, 856, 857, 859, 871, 895, 896, 899, 902, 908, 912, 918, 923, 924, 925, 950, 958, 965, 986, 995, 999, 1003, 1016, 1018, 1022, 1030, 1031, 1040, 1041, 1042, 1049, 1055, 1057, 1059, 1063, 1069, 1072, 1074, 1076, 1086, 1101, 1102, 1103, 1104, 1115, 1120, 1122, 1124, 1128, 1133, 1137, 1143, 1151, 1152, 1156, 1165, 1170, 1171

## 3. (8 puntos) Similitud de Coseno usando el indice invertido
Implementar búsqueda por similitud de coseno aprovechando el índice invertido:

- Proceso de búsqueda:
    - Recibe una consulta en lenguaje natural y un parámetro top_k
    - Utiliza el índice invertido para identificar documentos candidatos
    - Calcula similitud de coseno solo con los documentos relevantes utilizando los pesos TF-IDF
    - Retorna los top-k documentos más similares

<img src="https://1drv.ms/i/c/0c2923df9f1f816f/IQSELMi5qcbqS7lsy5sn8ZLpAZ3G2ciXdabecVJ0vhKoL78" width="500" align="" />

####  Pruebas funcionales

In [6]:
test_queries = [
    {
        "query": "¿Cuáles son las últimas innovaciones en la banca digital y la tecnología financiera?",
        "top_k": 5
    },
    {
        "query": "evolución de la inflación y el crecimiento de la economía en los últimos años",
        "top_k": 5
    },
    {
        "query": "avances sobre sostenibilidad y energías renovables para el medio ambiente",
        "top_k": 5
    }
]

for test in test_queries:    
    results = idx.cosine_search(test['query'], test['top_k'])

    print("\n=============================================================================================================================")
    print(f"Top {test['top_k']} documentos más similares:")    
    for doc_id, score in results:
        print(f"Doc {doc_id}: {score:.3f}: ", idx.showDocument2(doc_id))


Top 5 documentos más similares:
Doc 836: 0.121:  En medio de la apuesta de Bbva por la transformación digital de la banca la entidad financiera en Co...
Doc 1151: 0.110:  Cerca de 500 asistentes, entre empresas, inversores, fondos y socios de BBVA se han dado cita en una...
Doc 308: 0.109:  BBVA participó de Forbes Revolución Fintech Summit. El encuentro organizado por la Revista Forbes an...
Doc 1083: 0.109:  BBVA participó de Forbes Revolución Fintech Summit. El encuentro organizado por la Revista Forbes an...
Doc 1115: 0.108:  La innovación y necesidad de adecuarse a las nuevas tecnologías como una herramienta trascendental p...

Top 5 documentos más similares:
Doc 65: 0.171:  La economía global sigue dando señales crecientes de desaceleración. Las presiones inflacionistas po...
Doc 359: 0.171:  La economía global sigue dando señales crecientes de desaceleración. Las presiones inflacionistas po...
Doc 993: 0.171:  La economía global sigue dando señales crecientes de desaceleración.